# 06 — Tau Error Localization

Where in voltage space does the residual tau mismatch between the capped
Eyring model and classic HH live, and is the rate cap surgical or global?

Left: tau(V) for classic HH (black) and Eyring+cap (colored dashed); the
shaded band marks voltages where the rate cap is engaged. Right: signed
relative tau error, with rest / threshold / AP-peak landmarks and the peak
error marked. The summary table adds an AP-trajectory-weighted mean error
(the error the membrane actually experiences during a spike).

In [1]:
import sys; sys.path.insert(0, '/workspace')
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = ['Liberation Sans','Arimo','DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
FIG_DIR = '/mnt/results/hh_simulator/figures'
os.makedirs(FIG_DIR, exist_ok=True)
from hh_simulator import (presets, NaChannel, KChannel, LeakChannel,
    PointCell, Simulator, step_pulse)

In [2]:

V = np.linspace(-100, 50, 2001)
LANDMARKS = [(-65.0, 'rest'), (-55.0, 'thresh'), (40.0, 'peak')]
colors = {'m': '#0279EE', 'h': '#FD9BED', 'n': '#FF9400'}
labels = {'m': 'm (Na act.)', 'h': 'h (Na inact.)', 'n': 'n (K act.)'}

# Reference AP trajectory (classic mode) for trajectory-weighted error
cell = PointCell([NaChannel('classic'), KChannel('classic'), LeakChannel()])
sol = Simulator(cell).run((0, 40), I_inj=step_pulse((0, 40), 10.0, onset=5, dur=30),
                          mode='deterministic', t_eval=np.linspace(0, 40, 4001))

fig, axes = plt.subplots(3, 2, figsize=(11, 8.5), sharex=True)
rows = {}
for i, p in enumerate(('m', 'h', 'n')):
    a_c, b_c = presets.CLASSIC_RATES[p]
    tau_c = 1.0 / (a_c(V) + b_c(V))
    el = presets.fitted_landscape(p)
    tau_e = el.tau(V)
    rel = (tau_e - tau_c) / tau_c * 100.0

    # cap-active region (alpha()/beta() return capped rates)
    a_e, b_e = el.alpha(V), el.beta(V)
    cap_mask = np.zeros(V.shape, dtype=bool)
    if np.isfinite(el.alpha_cap):
        cap_mask |= a_e >= 0.999 * el.alpha_cap
    if np.isfinite(el.beta_cap):
        cap_mask |= b_e >= 0.999 * el.beta_cap
    v_cap = V[np.argmax(cap_mask)] if cap_mask.any() else np.nan

    axL, axR = axes[i]
    axL.semilogy(V, tau_c, 'k-', lw=1.5, label='classic HH')
    axL.semilogy(V, tau_e, color=colors[p], ls='--', lw=1.5, label='Eyring + cap')
    if np.isfinite(v_cap):
        for ax in (axL, axR):
            ax.axvspan(v_cap, V[-1], color=colors[p], alpha=0.10, lw=0)
    axL.set_ylabel(labels[p] + '\n' + r'$\tau$ (ms)')
    axL.legend(frameon=True, framealpha=0.95, edgecolor='none', fontsize=8, loc='lower left')

    axR.plot(V, rel, color=colors[p], lw=1.5)
    axR.axhline(0, color='k', lw=0.6)
    for vv, lab in LANDMARKS:
        axR.axvline(vv, color='gray', ls=':', lw=0.8)
    j = int(np.argmax(np.abs(rel)))
    axR.plot(V[j], rel[j], 'o', color='k', ms=4)
    axR.set_ylabel(r'$\tau$ error (%)')
    if i == 0:
        for vv, lab in LANDMARKS:
            axR.annotate(lab, (vv, axR.get_ylim()[1]), textcoords='offset points',
                         xytext=(3, -2), fontsize=7, color='gray', va='top')

    at = lambda vv: float(np.interp(vv, V, rel))
    sub = V < -55.0
    rel_ap = np.interp(sol.V, V, rel)          # error along the AP trajectory
    rows[p] = dict(
        v_cap=v_cap, vmax=V[j], emax=rel[j],
        e_rest=at(-65.0), e_thresh=at(-55.0), e_peak=at(40.0),
        m_sub=float(np.mean(np.abs(rel[sub]))),
        m_supra=float(np.mean(np.abs(rel[~sub]))),
        m_ap=float(np.mean(np.abs(rel_ap))),
        tau_c_rest=float(np.interp(-65.0, V, tau_c)),
    )

axes[2, 0].set_xlabel('V (mV)'); axes[2, 1].set_xlabel('V (mV)')
axes[0, 0].set_title('Time constants'); axes[0, 1].set_title('Signed relative error')
fig.suptitle('Tau error localization')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/06_tau_error_localization.svg', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/06_tau_error_localization.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"{'p':<2} {'cap_on':>7} {'V@maxE':>7} {'maxE%':>7} {'rest%':>7} {'thr%':>7} {'peak%':>7} "
      f"{'m|sub|':>7} {'m|sup|':>7} {'m|AP|':>7} {'tau_c(rest)':>11}")
for p, r in rows.items():
    print(f"{p:<2} {r['v_cap']:>7.1f} {r['vmax']:>7.1f} {r['emax']:>7.1f} {r['e_rest']:>7.1f} "
          f"{r['e_thresh']:>7.1f} {r['e_peak']:>7.1f} {r['m_sub']:>7.1f} {r['m_supra']:>7.1f} "
          f"{r['m_ap']:>7.2f} {r['tau_c_rest']:>11.3f}")
print('\nFraction of 40 ms AP trace spent with V > cap onset:')
for p, r in rows.items():
    frac = float(np.mean(sol.V > r['v_cap']))
    print(f'  {p}: V > {r["v_cap"]:.1f} mV for {frac*100:.1f}% of the trace')


p   cap_on  V@maxE   maxE%   rest%    thr%   peak%  m|sub|  m|sup|   m|AP| tau_c(rest)
m     18.3  -100.0   -49.8   -26.0   -17.1   -11.0    34.8    12.9   24.39       0.237
h    -30.2   -30.2   -37.6     3.6     7.4    -0.0     5.0     7.3    4.13       8.516
n      6.1     6.1   -38.8    17.8    24.4    -8.2    13.4    19.9   18.26       5.459

Fraction of 40 ms AP trace spent with V > cap onset:
  m: V > 18.3 mV for 3.1% of the trace
  h: V > -30.2 mV for 9.3% of the trace
  n: V > 6.1 mV for 4.5% of the trace
